# Transformar csv a parquet

In [5]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os

archivo_csv = "../1_data_processed/v3_dataset_post_chi.csv"
archivo_parquet = "../1_data_processed/v3_dataset_post_chi.parquet"

os.makedirs("../1_data_processed/", exist_ok=True)

# Si ya existe, lo sobreescribimos
if os.path.exists(archivo_parquet):
    os.remove(archivo_parquet)

# Leer encabezado para definir tipos compactos
cols = pd.read_csv(archivo_csv, nrows=0).columns.tolist()

dtype_map = {}
for c in cols:
    if c.startswith("signo_zodiacal_"):
        dtype_map[c] = "uint8"
    elif c == "ESTANCIA_DIAS":
        dtype_map[c] = "int32"
    else:
        dtype_map[c] = "uint8"

chunksize = 100_000
writer = None
total = 0

for chunk in pd.read_csv(archivo_csv, dtype=dtype_map, chunksize=chunksize, low_memory=False):
    table = pa.Table.from_pandas(chunk, preserve_index=False)

    if writer is None:
        writer = pq.ParquetWriter(
            archivo_parquet,
            table.schema,
            compression="snappy"
        )

    writer.write_table(table)
    total += len(chunk)
    print(f"✅ Chunk convertido: {chunk.shape} | acumulado filas: {total:,}")

if writer is not None:
    writer.close()

print("✅ Parquet generado en:", archivo_parquet)

✅ Chunk convertido: (100000, 488) | acumulado filas: 100,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 200,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 300,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 400,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 500,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 600,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 700,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 800,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 900,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 1,000,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 1,100,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 1,200,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 1,300,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 1,400,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 1,500,000
✅ Chunk convertido: (100000, 488) | acumulado filas: 1,600,000
✅ Chunk co

# Train/Test

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"
CARPETA_SPLIT = Path("../1_data_processed/split_indices")
CARPETA_SPLIT.mkdir(parents=True, exist_ok=True)

TEST_SIZE = 0.20
BASE_SEED = 42

# =========================================================
# CARGA
# =========================================================
for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]

if not cols_signo:
    raise ValueError("No se encontraron columnas target tipo 'signo_zodiacal_*'.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

label_multiclase = df[cols_signo].idxmax(axis=1)

from sklearn.model_selection import train_test_split

idx = np.arange(len(df))
idx_train, idx_test = train_test_split(
    idx,
    test_size=TEST_SIZE,
    random_state=BASE_SEED,
    stratify=label_multiclase
)

np.save(CARPETA_SPLIT / "idx_train.npy", idx_train)
np.save(CARPETA_SPLIT / "idx_test.npy", idx_test)

print("✅ Split guardado")
print("Train:", len(idx_train))
print("Test:", len(idx_test))

Dataset cargado: (5808498, 488)
✅ Split guardado
Train: 4646798
Test: 1161700


# Modelado LR

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"
RUTA_SPLIT = Path("../1_data_processed/split_indices")

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_lr")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

LR_PARAMS = dict(
    solver="liblinear",
    C=1.0,
    max_iter=2000,
    random_state=BASE_SEED,
    class_weight="balanced"
)

MODELO = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(**LR_PARAMS))
])

# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan
    try:
        if hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

# =========================================================
# CARGA
# =========================================================
for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]
if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

# Compactar tipos
for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

idx_train = np.load(RUTA_SPLIT / "idx_train.npy")

# Pasar a NumPy para acelerar el loop
X_all = df.drop(columns=cols_signo).to_numpy(copy=False)
df_train = df.iloc[idx_train].reset_index(drop=True)
X_train_all = X_all[idx_train]

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

# =========================================================
# RUN
# =========================================================
resultados = []

for signo in cols_signo:
    print(f"\n🔮 LR -> {signo}")

    y_train_all = df_train[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"⚠️ Saltando {signo}")
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        # Bootstrap balanceado
        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        # CV k=5
        for tr_idx, val_idx in skf.split(X_boot, y_boot):
            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({
            "modelo": "lr",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
            "cv_precision": np.nanmean(fold_metrics[:, 1]),
            "cv_recall": np.nanmean(fold_metrics[:, 2]),
            "cv_f1": np.nanmean(fold_metrics[:, 3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
            "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:, 9]),
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

    # Guardado parcial por signo
    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"lr_detalle_{signo}.csv", index=False)
    resultados.extend(resultados_signo)

# =========================================================
# GUARDADO FINAL
# =========================================================
df_detalle = pd.DataFrame(resultados)
detalle_path = CARPETA_OUT / "lr_detalle.csv"
resumen_path = CARPETA_OUT / "lr_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc", "cv_pr_auc",
    "cv_tp_mean", "cv_fp_mean", "cv_tn_mean", "cv_fn_mean"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )
    df_resumen.to_csv(resumen_path)
    print("✅ LR terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)
else:
    print("⚠️ No hubo resultados para guardar.")

Dataset cargado: (5808498, 488)

🔮 LR -> signo_zodiacal_acuario
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🔮 LR -> signo_zodiacal_aries
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🔮 LR -> signo_zodiacal_capricornio
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500

# Modelado RF

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"
RUTA_SPLIT = Path("../1_data_processed/split_indices")

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_rf")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

MODELO = RandomForestClassifier(
    n_estimators=100,      # puedes subir a 200 después si quieres
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    n_jobs=-1,             # usar todos los cores
    random_state=BASE_SEED,
    class_weight="balanced"
)

# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan
    try:
        y_proba = modelo.predict_proba(X_eval)[:, 1]
        roc = roc_auc_score(y_eval, y_proba)
        pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

# =========================================================
# CARGA
# =========================================================
for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]
if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

idx_train = np.load(RUTA_SPLIT / "idx_train.npy")

X_all = df.drop(columns=cols_signo).to_numpy(copy=False)
df_train = df.iloc[idx_train].reset_index(drop=True)

X_train_all = X_all[idx_train]

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

# =========================================================
# RUN
# =========================================================
resultados = []

for signo in cols_signo:
    print(f"\n🔮 RF -> {signo}")

    y_train_all = df_train[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"⚠️ Saltando {signo}")
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        for tr_idx, val_idx in skf.split(X_boot, y_boot):
            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({
            "modelo": "rf",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
            "cv_precision": np.nanmean(fold_metrics[:, 1]),
            "cv_recall": np.nanmean(fold_metrics[:, 2]),
            "cv_f1": np.nanmean(fold_metrics[:, 3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
            "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:, 9]),
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

    # guardar parcial por signo
    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"rf_detalle_{signo}.csv", index=False)
    resultados.extend(resultados_signo)

# =========================================================
# GUARDADO FINAL
# =========================================================
df_detalle = pd.DataFrame(resultados)
detalle_path = CARPETA_OUT / "rf_detalle.csv"
resumen_path = CARPETA_OUT / "rf_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc", "cv_pr_auc",
    "cv_tp_mean", "cv_fp_mean", "cv_tn_mean", "cv_fn_mean"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )
    df_resumen.to_csv(resumen_path)
    print("✅ RF terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)
else:
    print("⚠️ No hubo resultados para guardar.")

Dataset cargado: (5808498, 488)

🔮 RF -> signo_zodiacal_acuario
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🔮 RF -> signo_zodiacal_aries
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500
   Iter 400/500
   Iter 425/500
   Iter 450/500
   Iter 475/500
   Iter 500/500

🔮 RF -> signo_zodiacal_capricornio
   Iter 25/500
   Iter 50/500
   Iter 75/500
   Iter 100/500
   Iter 125/500
   Iter 150/500
   Iter 175/500
   Iter 200/500
   Iter 225/500
   Iter 250/500
   Iter 275/500
   Iter 300/500
   Iter 325/500
   Iter 350/500
   Iter 375/500

# Modelado DT

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"
RUTA_SPLIT = Path("../1_data_processed/split_indices")

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_dt")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

MODELO = DecisionTreeClassifier(
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=BASE_SEED,
    class_weight="balanced"
)

# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan
    try:
        if hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:, 1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

# =========================================================
# CARGA
# =========================================================
for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]
if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

# Compactar tipos
for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

idx_train = np.load(RUTA_SPLIT / "idx_train.npy")

# Pasar a NumPy para acelerar el loop
X_all = df.drop(columns=cols_signo).to_numpy(copy=False)
df_train = df.iloc[idx_train].reset_index(drop=True)
X_train_all = X_all[idx_train]

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

# =========================================================
# RUN
# =========================================================
resultados = []

for signo in cols_signo:
    print(f"\n🔮 DT -> {signo}")

    y_train_all = df_train[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"⚠️ Saltando {signo}")
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        # Bootstrap balanceado
        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        # CV k=5
        for tr_idx, val_idx in skf.split(X_boot, y_boot):
            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({
            "modelo": "dt",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": float(y_boot.mean()),
            "cv_accuracy": np.nanmean(fold_metrics[:, 0]),
            "cv_precision": np.nanmean(fold_metrics[:, 1]),
            "cv_recall": np.nanmean(fold_metrics[:, 2]),
            "cv_f1": np.nanmean(fold_metrics[:, 3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:, 4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:, 5]),
            "cv_tp_mean": np.nanmean(fold_metrics[:, 6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:, 7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:, 8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:, 9]),
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

    # Guardado parcial por signo
    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"dt_detalle_{signo}.csv", index=False)
    resultados.extend(resultados_signo)

# =========================================================
# GUARDADO FINAL
# =========================================================
df_detalle = pd.DataFrame(resultados)
detalle_path = CARPETA_OUT / "dt_detalle.csv"
resumen_path = CARPETA_OUT / "dt_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc", "cv_pr_auc",
    "cv_tp_mean", "cv_fp_mean", "cv_tn_mean", "cv_fn_mean"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )
    df_resumen.to_csv(resumen_path)
    print("✅ DT terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)
else:
    print("⚠️ No hubo resultados para guardar.")


🔮 DT -> signo_zodiacal_acuario
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

🔮 DT -> signo_zodiacal_aries
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

🔮 DT -> signo_zodiacal_capricornio
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

🔮 DT -> signo_zodiacal_cancer
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

🔮 DT -> signo_zodiacal_escorpio
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

🔮 DT -> signo_zodiacal_geminis
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

🔮 DT -> signo_zodiacal_leo
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

🔮 DT -> signo_zodiacal_libra
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

🔮 DT -> signo_zodiacal_piscis
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

🔮 DT -> signo_zodiacal_sagitario
   Iter 25/100
   Iter 50/100
   Iter 75/100
   Iter 100/100

🔮 DT -> signo_zodiacal_tauro
   Iter 25/100
   Iter 50/100
   Iter 75/10

TypeError: agg function failed [how->mean,dtype->object]

# Modelado Knn

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"
RUTA_SPLIT = Path("../1_data_processed/split_indices")

N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_knn")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

MODELO = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(
        n_neighbors=50,
        weights="uniform",
        metric="euclidean",
        n_jobs=1
    ))
])

# =========================================================
# HELPERS
# =========================================================
def evaluar_modelo(modelo, X_eval, y_eval):

    y_pred = modelo.predict(X_eval)

    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan

    try:
        if hasattr(modelo, "predict_proba"):
            y_proba = modelo.predict_proba(X_eval)[:,1]
            roc = roc_auc_score(y_eval, y_proba)
            pr = average_precision_score(y_eval, y_proba)
    except:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

# =========================================================
# CARGA
# =========================================================
for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]

if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

# compactar tipos
for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

idx_train = np.load(RUTA_SPLIT / "idx_train.npy")

# usar numpy para acelerar
X_all = df.drop(columns=cols_signo).to_numpy(copy=False)

df_train = df.iloc[idx_train].reset_index(drop=True)
X_train_all = X_all[idx_train]

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

# =========================================================
# RUN
# =========================================================
resultados = []

for signo in cols_signo:

    print(f"\n🔮 KNN -> {signo}")

    y_train_all = df_train[signo].to_numpy(dtype=np.uint8)

    pos_idx = np.where(y_train_all == 1)[0]
    neg_idx = np.where(y_train_all == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"⚠️ Saltando {signo}")
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    resultados_signo = []

    for it in range(N_ITER):

        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)

        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all[sample_idx]
        y_boot = y_train_all[sample_idx]

        fold_metrics = []

        for tr_idx, val_idx in skf.split(X_boot, y_boot):

            X_tr, X_val = X_boot[tr_idx], X_boot[val_idx]
            y_tr, y_val = y_boot[tr_idx], y_boot[val_idx]

            m_cv = clone(MODELO)
            m_cv.fit(X_tr, y_tr)

            fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

        fold_metrics = np.array(fold_metrics, dtype=float)

        resultados_signo.append({

            "modelo": "knn",
            "signo": signo,
            "iter": it + 1,

            "pos_rate_bootstrap": float(y_boot.mean()),

            "cv_accuracy": np.nanmean(fold_metrics[:,0]),
            "cv_precision": np.nanmean(fold_metrics[:,1]),
            "cv_recall": np.nanmean(fold_metrics[:,2]),
            "cv_f1": np.nanmean(fold_metrics[:,3]),
            "cv_roc_auc": np.nanmean(fold_metrics[:,4]),
            "cv_pr_auc": np.nanmean(fold_metrics[:,5]),

            "cv_tp_mean": np.nanmean(fold_metrics[:,6]),
            "cv_fp_mean": np.nanmean(fold_metrics[:,7]),
            "cv_tn_mean": np.nanmean(fold_metrics[:,8]),
            "cv_fn_mean": np.nanmean(fold_metrics[:,9])
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

    df_signo = pd.DataFrame(resultados_signo)
    df_signo.to_csv(CARPETA_OUT / f"knn_detalle_{signo}.csv", index=False)

    resultados.extend(resultados_signo)

# =========================================================
# GUARDADO FINAL
# =========================================================
df_detalle = pd.DataFrame(resultados)

detalle_path = CARPETA_OUT / "knn_detalle.csv"
resumen_path = CARPETA_OUT / "knn_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
"pos_rate_bootstrap",
"cv_accuracy","cv_precision","cv_recall","cv_f1","cv_roc_auc","cv_pr_auc",
"cv_tp_mean","cv_fp_mean","cv_tn_mean","cv_fn_mean"
]

if not df_detalle.empty:

    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean","std"])
    )

    df_resumen.to_csv(resumen_path)

    print("✅ KNN terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)

else:
    print("⚠️ No hubo resultados.")

# Modelado SVM

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.svm import SVC

# =========================================================
# CONFIG
# =========================================================
RUTA_DATASET = "../1_data_processed/v3_dataset_post_chi.parquet"
RUTA_SPLIT = Path("../1_data_processed/split_indices")

USE_CV = True
N_SAMPLE = 100
N_ITER = 500
K_FOLDS = 5
BASE_SEED = 42
PROGRESS_EVERY = 25

CARPETA_OUT = Path("../4_results/modelo_svm")
CARPETA_OUT.mkdir(parents=True, exist_ok=True)

MODELO = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="linear",
        C=1.0,
        probability=True,
        random_state=BASE_SEED,
        class_weight="balanced"
    ))
])

def evaluar_modelo(modelo, X_eval, y_eval):
    y_pred = modelo.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred, labels=[0, 1]).ravel()

    acc = accuracy_score(y_eval, y_pred)
    prec = precision_score(y_eval, y_pred, zero_division=0)
    rec = recall_score(y_eval, y_pred, zero_division=0)
    f1 = f1_score(y_eval, y_pred, zero_division=0)

    roc = np.nan
    pr = np.nan
    try:
        y_proba = modelo.predict_proba(X_eval)[:, 1]
        roc = roc_auc_score(y_eval, y_proba)
        pr = average_precision_score(y_eval, y_proba)
    except Exception:
        pass

    return acc, prec, rec, f1, roc, pr, tp, fp, tn, fn

for ext_name in ["pandas.period", "pandas.interval"]:
    try:
        pa.unregister_extension_type(ext_name)
    except Exception:
        pass

cols = pq.read_schema(RUTA_DATASET).names
cols_signo = [c for c in cols if c.startswith("signo_zodiacal_")]
if not cols_signo:
    raise ValueError("No se encontraron columnas signo_zodiacal_*.")

df = pd.read_parquet(RUTA_DATASET, engine="pyarrow")
print("Dataset cargado:", df.shape)

for c in df.columns:
    if c.startswith("signo_zodiacal_"):
        df[c] = df[c].astype("uint8")
    elif c == "ESTANCIA_DIAS":
        df[c] = df[c].astype("int32")
    elif pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].astype("uint8")

idx_train = np.load(RUTA_SPLIT / "idx_train.npy")
idx_test = np.load(RUTA_SPLIT / "idx_test.npy")

X_all = df.drop(columns=cols_signo).copy()
X_train_all = X_all.iloc[idx_train].reset_index(drop=True)
X_test_all = X_all.iloc[idx_test].reset_index(drop=True)
df_train = df.iloc[idx_train].reset_index(drop=True)
df_test = df.iloc[idx_test].reset_index(drop=True)

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=BASE_SEED)

resultados = []

for signo in cols_signo:
    print(f"\n🔮 SVM -> {signo}")

    y_train_all = df_train[signo].astype(int)
    y_test_all = df_test[signo].astype(int)

    pos_idx = np.where(y_train_all.values == 1)[0]
    neg_idx = np.where(y_train_all.values == 0)[0]

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        print(f"⚠️ Saltando {signo}")
        continue

    n_pos = N_SAMPLE // 2
    n_neg = N_SAMPLE - n_pos

    for it in range(N_ITER):
        rng = np.random.default_rng(BASE_SEED + it + abs(hash(signo)) % 10000)

        sample_pos = rng.choice(pos_idx, size=n_pos, replace=True)
        sample_neg = rng.choice(neg_idx, size=n_neg, replace=True)
        sample_idx = np.concatenate([sample_pos, sample_neg])
        rng.shuffle(sample_idx)

        X_boot = X_train_all.iloc[sample_idx]
        y_boot = y_train_all.iloc[sample_idx]

        pos_rate = float(y_boot.mean())

        if USE_CV:
            fold_metrics = []

            for tr_idx, val_idx in skf.split(X_boot, y_boot):
                X_tr, X_val = X_boot.iloc[tr_idx], X_boot.iloc[val_idx]
                y_tr, y_val = y_boot.iloc[tr_idx], y_boot.iloc[val_idx]

                m_cv = clone(MODELO)
                m_cv.fit(X_tr, y_tr)
                fold_metrics.append(evaluar_modelo(m_cv, X_val, y_val))

            fold_metrics = np.array(fold_metrics, dtype=float)

            cv_accuracy = np.nanmean(fold_metrics[:, 0])
            cv_precision = np.nanmean(fold_metrics[:, 1])
            cv_recall = np.nanmean(fold_metrics[:, 2])
            cv_f1 = np.nanmean(fold_metrics[:, 3])
            cv_roc_auc = np.nanmean(fold_metrics[:, 4])
            cv_pr_auc = np.nanmean(fold_metrics[:, 5])
        else:
            cv_accuracy = np.nan
            cv_precision = np.nan
            cv_recall = np.nan
            cv_f1 = np.nan
            cv_roc_auc = np.nan
            cv_pr_auc = np.nan

        m_test = clone(MODELO)
        m_test.fit(X_boot, y_boot)

        test_acc, test_prec, test_rec, test_f1, test_roc, test_pr, test_tp, test_fp, test_tn, test_fn = evaluar_modelo(
            m_test, X_test_all, y_test_all
        )

        resultados.append({
            "modelo": "svm",
            "signo": signo,
            "iter": it + 1,
            "pos_rate_bootstrap": pos_rate,
            "cv_accuracy": cv_accuracy,
            "cv_precision": cv_precision,
            "cv_recall": cv_recall,
            "cv_f1": cv_f1,
            "cv_roc_auc": cv_roc_auc,
            "cv_pr_auc": cv_pr_auc,
            "test_accuracy": test_acc,
            "test_precision": test_prec,
            "test_recall": test_rec,
            "test_f1": test_f1,
            "test_roc_auc": test_roc,
            "test_pr_auc": test_pr,
            "test_tp": test_tp,
            "test_fp": test_fp,
            "test_tn": test_tn,
            "test_fn": test_fn
        })

        if (it + 1) % PROGRESS_EVERY == 0:
            print(f"   Iter {it+1}/{N_ITER}")

df_detalle = pd.DataFrame(resultados)
detalle_path = CARPETA_OUT / "svm_detalle.csv"
resumen_path = CARPETA_OUT / "svm_resumen.csv"

df_detalle.to_csv(detalle_path, index=False)

cols_metricas = [
    "pos_rate_bootstrap",
    "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "cv_roc_auc", "cv_pr_auc",
    "test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc", "test_pr_auc",
    "test_tp", "test_fp", "test_tn", "test_fn"
]

if not df_detalle.empty:
    df_resumen = (
        df_detalle
        .groupby("signo")[cols_metricas]
        .agg(["mean", "std"])
    )
    df_resumen.to_csv(resumen_path)
    print("✅ SVM terminado")
    print("Detalle:", detalle_path)
    print("Resumen:", resumen_path)
else:
    print("⚠️ No hubo resultados para guardar.")

# Mostrar Resultados

In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# =====================================================
# RUTAS DE LOS RESÚMENES
# =====================================================
BASE_PATH = Path("../4_results")

resumenes = {
    "Random Forest": BASE_PATH / "modelo_rf" / "rf_resumen.csv",
    "Árbol de Decisión": BASE_PATH / "modelo_dt" / "dt_resumen.csv",
    "KNN": BASE_PATH / "modelo_knn" / "knn_resumen.csv",
    "Regresión Logística": BASE_PATH / "modelo_lr" / "lr_resumen.csv",
    "SVM": BASE_PATH / "modelo_svm" / "svm_resumen.csv",
}

medias_modelos = {}

# =====================================================
# CARGA Y PROCESAMIENTO
# =====================================================
for nombre_modelo, ruta in resumenes.items():
    print(f"\n📊 {nombre_modelo}")
    print("-" * 50)

    if not ruta.exists():
        print(f"⚠️ No se encontró el archivo: {ruta}")
        continue

    df_resumen = pd.read_csv(ruta, header=[0, 1], index_col=0)

    # Extraer solo las medias
    df_medias = df_resumen.xs("mean", axis=1, level=1)

    # Redondear
    df_medias = df_medias.round(3)

    medias_modelos[nombre_modelo] = df_medias

    display(df_medias.style.format("{:.3f}"))



📊 Random Forest
--------------------------------------------------
⚠️ No se encontró el archivo: ..\4_results\modelo_rf\rf_resumen.csv

📊 Árbol de Decisión
--------------------------------------------------


,pos_rate_bootstrap,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,cv_pr_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc,test_tp,test_fp,test_tn,test_fn
signo,,,,,,,,,,,,,,,,,
signo_zodiacal_acuario,0.500,0.503,0.504,0.495,0.491,0.503,0.514,0.502,0.079,0.497,0.136,0.500,0.079,45444.860,532207.700,538130.300,45917.140
signo_zodiacal_aries,0.500,0.504,0.505,0.502,0.495,0.503,0.515,0.497,0.076,0.504,0.132,0.500,0.076,44343.370,540960.100,532801.900,43594.630
signo_zodiacal_cancer,0.500,0.495,0.494,0.492,0.485,0.496,0.510,0.502,0.085,0.498,0.144,0.500,0.085,48890.680,529449.730,533994.270,49365.320
signo_zodiacal_capricornio,0.500,0.503,0.502,0.488,0.487,0.502,0.514,0.505,0.081,0.495,0.140,0.500,0.081,46800.450,527649.100,539468.900,47781.550
signo_zodiacal_escorpio,0.500,0.497,0.496,0.496,0.487,0.497,0.513,0.500,0.084,0.501,0.144,0.500,0.084,48840.360,532261.870,531866.130,48731.640
signo_zodiacal_geminis,0.500,0.494,0.494,0.493,0.484,0.494,0.510,0.498,0.084,0.503,0.144,0.500,0.084,48955.880,534507.620,529803.380,48433.120
signo_zodiacal_leo,0.500,0.508,0.509,0.506,0.499,0.509,0.516,0.498,0.086,0.502,0.146,0.500,0.086,49947.620,533382.320,528794.680,49575.380
signo_zodiacal_libra,0.500,0.500,0.504,0.502,0.494,0.499,0.514,0.506,0.092,0.492,0.155,0.500,0.092,52651.670,519068.000,535679.000,54301.330
signo_zodiacal_piscis,0.500,0.499,0.498,0.488,0.486,0.498,0.512,0.505,0.079,0.494,0.136,0.500,0.079,45417.710,528718.530,541106.470,46457.290



📊 KNN
--------------------------------------------------
⚠️ No se encontró el archivo: ..\4_results\modelo_knn\knn_resumen.csv

📊 Regresión Logística
--------------------------------------------------


,pos_rate_bootstrap,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_roc_auc,cv_pr_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc,test_tp,test_fp,test_tn,test_fn
signo,,,,,,,,,,,,,,,,,
signo_zodiacal_acuario,0.500,nan,nan,nan,nan,nan,nan,0.506,0.079,0.493,0.136,0.500,0.079,45077.680,527527.260,542810.740,46284.320
signo_zodiacal_aries,0.500,nan,nan,nan,nan,nan,nan,0.502,0.076,0.498,0.131,0.500,0.076,43816.400,534590.760,539171.240,44121.600
signo_zodiacal_cancer,0.500,nan,nan,nan,nan,nan,nan,0.503,0.085,0.497,0.144,0.500,0.085,48799.680,527854.300,535589.700,49456.320
signo_zodiacal_capricornio,0.500,nan,nan,nan,nan,nan,nan,0.503,0.081,0.496,0.140,0.500,0.082,46924.670,529184.320,537933.680,47657.330
signo_zodiacal_escorpio,0.500,nan,nan,nan,nan,nan,nan,0.505,0.084,0.495,0.144,0.501,0.084,48267.120,525243.950,538884.050,49304.880
signo_zodiacal_geminis,0.500,nan,nan,nan,nan,nan,nan,0.495,0.084,0.507,0.144,0.500,0.084,49349.010,538866.100,525444.900,48039.990
signo_zodiacal_leo,0.500,nan,nan,nan,nan,nan,nan,0.498,0.086,0.502,0.146,0.500,0.086,49958.100,533298.310,528878.690,49564.900
signo_zodiacal_libra,0.500,nan,nan,nan,nan,nan,nan,0.503,0.092,0.497,0.156,0.501,0.092,53203.810,523479.620,531267.380,53749.190
signo_zodiacal_piscis,0.500,nan,nan,nan,nan,nan,nan,0.499,0.079,0.501,0.136,0.500,0.079,46016.020,535999.620,533825.380,45858.980



📊 SVM
--------------------------------------------------
⚠️ No se encontró el archivo: ..\4_results\modelo_svm\svm_resumen.csv


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# =====================================================
# CONFIGURACIÓN
# =====================================================
BASE_PATH = Path("../4_results")

MODELOS = {
    "Random Forest": BASE_PATH / "modelo_rf" / "rf_detalle.csv",
    "Árbol de Decisión": BASE_PATH / "modelo_dt" / "dt_detalle.csv",
    "KNN": BASE_PATH / "modelo_knn" / "knn_detalle.csv",
    "Regresión Logística": BASE_PATH / "modelo_lr" / "lr_detalle.csv",
    "SVM": BASE_PATH / "modelo_svm" / "svm_detalle.csv",
}

signos = [
    'signo_zodiacal_acuario',
    'signo_zodiacal_aries',
    'signo_zodiacal_capricornio',
    'signo_zodiacal_cancer',
    'signo_zodiacal_escorpio',
    'signo_zodiacal_geminis',
    'signo_zodiacal_leo',
    'signo_zodiacal_libra',
    'signo_zodiacal_piscis',
    'signo_zodiacal_sagitario',
    'signo_zodiacal_tauro',
    'signo_zodiacal_virgo'
]

CARPETA_FIG = Path("../4_results/figures_modelado")
CARPETA_FIG.mkdir(parents=True, exist_ok=True)

METRICA = "cv_roc_auc"   # cambiar a "cv_roc_auc" si quieres CV

# =====================================================
# FUNCIÓN DE GRÁFICO
# =====================================================
def plot_roc_auc_4x3(df, nombre_modelo, output_path, metrica):
    fig, axes = plt.subplots(4, 3, figsize=(16, 10), sharey=True)
    axes = axes.flatten()

    for ax, signo in zip(axes, signos):
        df_s = df[df["signo"] == signo].sort_values("iter")

        if df_s.empty or metrica not in df_s.columns:
            ax.set_title(f"{signo.replace('signo_zodiacal_', '').capitalize()} (sin datos)")
            ax.axis("off")
            continue

        media = df_s[metrica].mean()

        ax.scatter(df_s["iter"], df_s[metrica], s=12, alpha=0.5)
        ax.axhline(0.5, color="red", linestyle="--", linewidth=1)
        ax.axhline(media, color="blue", linewidth=1.5)

        ax.set_title(
            f"{signo.replace('signo_zodiacal_', '').capitalize()} (μ = {media:.3f})",
            fontsize=11
        )

        ax.set_ylim(0.3, 0.7)
        ax.set_xlabel("Iteración")
        ax.set_ylabel("ROC-AUC")

    plt.suptitle(
        f"Distribución de {metrica} por iteración\n({nombre_modelo})",
        fontsize=15
    )

    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    plt.savefig(output_path, dpi=300)
    plt.close()

# =====================================================
# EJECUCIÓN POR MODELO
# =====================================================
for nombre_modelo, ruta_csv in MODELOS.items():
    print(f"📊 Generando figura para {nombre_modelo}")

    if not ruta_csv.exists():
        print(f"   ⚠️ No existe: {ruta_csv}")
        continue

    df = pd.read_csv(ruta_csv)

    out_img = CARPETA_FIG / f"{METRICA}_4x3_{nombre_modelo.replace(' ', '_').lower()}.png"
    plot_roc_auc_4x3(df, nombre_modelo, out_img, METRICA)

    print(f"   ✅ Guardado en: {out_img}")

print("\n✅ Todas las figuras generadas")

📊 Generando figura para Random Forest
   ✅ Guardado en: ..\4_results\figures_modelado\cv_roc_auc_4x3_random_forest.png
📊 Generando figura para Árbol de Decisión
   ⚠️ No existe: ..\4_results\modelo_dt\dt_detalle.csv
📊 Generando figura para KNN
   ⚠️ No existe: ..\4_results\modelo_knn\knn_detalle.csv
📊 Generando figura para Regresión Logística
   ✅ Guardado en: ..\4_results\figures_modelado\cv_roc_auc_4x3_regresión_logística.png
📊 Generando figura para SVM
   ⚠️ No existe: ..\4_results\modelo_svm\svm_detalle.csv

✅ Todas las figuras generadas
